## **MLlib: 영화 추천 알고리즘 구현**
+ 데이터셋: MovieLens(2500만개 영화 평점 데이터), 본 예제에서는 7만개의 데이터만 사용 ```ratings_short.csv```
+ 추천 알고리즘: Alternating Least Squares (ALS)

---

### **1. 영화 평점 데이터 불러오기 & 데이터프레임 생성하기**

In [1]:
# [+] SparkSession 설정
from pyspark.sql import SparkSession
spark = SparkSession.builder.master('local').appName('recommandation').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 04:35:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


#### movielens 데이터 불러오기
+ 파일명: ```ratings_short.csv```
+ 스키마 설정: ```inferSchema=True```
+ 헤더 사용: ```header=True```

In [2]:
# movielens 데이터 불러오기
path = '../data/'
file = 'ratings_short.csv'

ratings_df = spark.read.csv(
    path + file, 
    inferSchema=True, 
    header=True
)

In [3]:
# 데이터프레임 출력
ratings_df.show()

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
|     1|   1088|   4.0|1147868495|
|     1|   1175|   3.5|1147868826|
|     1|   1217|   3.5|1147878326|
|     1|   1237|   5.0|1147868839|
|     1|   1250|   4.0|1147868414|
|     1|   1260|   3.5|1147877857|
|     1|   1653|   4.0|1147868097|
|     1|   2011|   2.5|1147868079|
|     1|   2012|   2.5|1147868068|
|     1|   2068|   2.5|1147869044|
|     1|   2161|   3.5|1147868609|
|     1|   2351|   4.5|1147877957|
|     1|   2573|   4.0|1147878923|
|     1|   2632|   5.0|1147878248|
|     1|   2692|   5.0|1147869100|
+------+-------+------+----------+
only showing top 20 rows



In [4]:
# 데이터프레임 스키마 출력
ratings_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [6]:
# [+] 타임스탬프를 제외한 컬럼 선택
ratings_df = ratings_df.select(['userId', 'movieId','rating'])

In [10]:
# [+] describe(): 기술 통계량 출력
ratings_df.select(['rating']).describe().show()

+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|             71921|
|   mean|3.5821387355570695|
| stddev| 1.042406032579843|
|    min|               0.5|
|    max|               5.0|
+-------+------------------+



### **2. 훈련 데이터 준비 및 영화 추천 모델 학습**

In [11]:
# [+] 훈련 데이터셋과 테스트 데이터셋 분리 (8:2)
train_df, test_df = ratings_df.randomSplit([0.8,0.2])

In [12]:
# 추천 알고리즘(Alternating Least Squares) 불러오기
from  pyspark.ml.recommendation import ALS

#### **ALS 알고리즘 초매개변수**
+ ```maxIter```: 최대 학습 반복 횟수
+ ```regParam```: 정규화 매개변수(범위: 0~1)
+ ```coldStartStrategy```: 데이터가 부족한 신규 유저 및 아이템에 대한 예측 문제(Cold Start)를 처리하는 방식이며 ```drop```값은 해당 데이터를 모델 학습 과정에서 배제

In [13]:
# 추천 알고리즘 초매개변수 설정
als = ALS(
    maxIter=5,
    regParam=0.1,
    userCol='userId',
    itemCol='movieId',
    ratingCol='rating',
    coldStartStrategy='drop'
)

In [14]:
# [+] 모델 학습
# estimater는 fit 메소드 사용
model = als.fit(train_df)

26/11/26 04:48:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/11/26 04:48:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/11/26 04:48:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


In [ ]:
# # 메모리 부족으로 인한 오류 발생시, 아래의 코드를 실행
# from pyspark.sql import SparkSession

# MAX_MEMORY = '5g'
# spark = SparkSession.builder.appName('movie-recommendation')\
#     .config('spark.executor.memory', MAX_MEMORY)\
#     .config('spark.driver.memory', MAX_MEMORY)\
#     .getOrCreate()

In [15]:
# [+] 모델 예측
predictions = model.transform(test_df)

In [16]:
# 예측값 출력
predictions.show()

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|    293|   4.0|  4.054508|
|   148|    527|   5.0| 4.1131873|
|   148|    903|   4.0| 4.1652923|
|   148|    908|   4.0| 4.2289352|
|   148|    922|   4.0|  3.882342|
|   148|   1080|   3.5|  4.179219|
|   148|   1086|   4.0| 4.1382923|
|   148|   1136|   4.5| 4.2561307|
|   148|   1178|   5.0| 4.0120916|
|   148|   1203|   5.0|  4.122928|
|   148|   1221|   4.5|  4.424764|
|   148|   1228|   4.5| 4.0041437|
|   148|   1260|   4.0| 3.6977434|
|   148|   1276|   3.5| 3.8790417|
|   148|   2176|   4.5| 4.0218897|
|   148|   3000|   4.0|   4.55172|
|   148|   3435|   4.5|  3.954859|
|   148|   4993|   4.5|  4.131485|
|   148|   5060|   4.0| 4.0253134|
|   148|  77455|   3.5| 3.1152842|
+------+-------+------+----------+
only showing top 20 rows



In [17]:
# [+] 평점과 예측평점에 대한 통계 출력
predictions.select(['rating','prediction']).describe().show()

+-------+------------------+------------------+
|summary|            rating|        prediction|
+-------+------------------+------------------+
|  count|             13696|             13696|
|   mean|3.5922532126168223|3.4299061318748483|
| stddev| 1.039338967114266|0.7517466427378738|
|    min|               0.5|        0.17665374|
|    max|               5.0|          5.657744|
+-------+------------------+------------------+



In [18]:
# 모델 성능 평가: RMSE(Root Mean Squared Error)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName='rmse', #mae: 평균절차오류? 0.7 나옴
    labelCol='rating',
    predictionCol='prediction'
)

In [19]:
# [+] RMSE 측정
rmse = evaluator.evaluate(predictions)

In [20]:
rmse

0.9122859362088975

### **3. 학습된 모델을 이용한 영화 추천**
+ ```recommendForAllUsers()```: 유저별 아이템 추천
+ ```recommendForAllItems()```: 아이템별 유저 추천
+ ```recommendForUserSubset()```: 특정 유저 그룹에 대한 아이템 추천

In [23]:
# [+] 유저별 아이템을 3개씩 추천
model.recommendForAllUsers(3).show()

[Stage 164:=================================================>    (91 + 1) / 100]

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{69699, 5.407512...|
|     2|[{8235, 6.0544705...|
|     3|[{123288, 4.99858...|
|     4|[{482, 5.0731125}...|
|     5|[{123288, 5.08822...|
|     6|[{735, 5.3811035}...|
|     7|[{60103, 4.891307...|
|     8|[{7156, 5.18024},...|
|     9|[{60103, 5.676620...|
|    10|[{170355, 5.05603...|
|    11|[{3179, 4.5815506...|
|    12|[{5690, 4.8706203...|
|    13|[{1301, 4.957908}...|
|    14|[{171011, 5.59654...|
|    15|[{117364, 5.68833...|
|    16|[{123288, 5.33051...|
|    17|[{3179, 5.0345273...|
|    18|[{78218, 4.767971...|
|    19|[{109364, 4.97282...|
|    20|[{1301, 5.5944953...|
+------+--------------------+
only showing top 20 rows



In [24]:
# [+] 아이템별 유저를 3명씩 추천
model.recommendForAllItems(3).show()

[Stage 193:====================================================> (98 + 1) / 100]

+-------+--------------------+
|movieId|     recommendations|
+-------+--------------------+
|      1|[{240, 5.2964897}...|
|      2|[{327, 4.704262},...|
|      3|[{198, 4.7671456}...|
|      4|[{240, 4.5368385}...|
|      5|[{87, 4.4089704},...|
|      6|[{87, 5.2023005},...|
|      7|[{329, 4.5305963}...|
|      8|[{240, 4.608613},...|
|      9|[{198, 4.2587543}...|
|     10|[{87, 4.4787035},...|
|     11|[{240, 4.9840727}...|
|     12|[{198, 4.8830924}...|
|     13|[{327, 5.597861},...|
|     14|[{463, 4.7979746}...|
|     15|[{371, 4.431196},...|
|     16|[{153, 5.3225756}...|
|     17|[{49, 5.021366}, ...|
|     18|[{240, 5.4405785}...|
|     19|[{56, 4.398566}, ...|
|     20|[{484, 4.62763}, ...|
+-------+--------------------+
only showing top 20 rows



**특정 유저 그룹에 대한 아이템 추천**
1. 해당 유저들의 ID 정보를 데이터프레임으로 생성
2. ```recommendForUserSubset()```를 이용한 아이템 추천

In [31]:
# 특정 유저 ID 집합: 2차원 리스트
user_lst = [[1]]

In [30]:
# 스키마 정의
from pyspark.sql.types import StructType, StructField, IntegerType
schema = StructType([
    StructField('userID', IntegerType(), True)
])

In [32]:
# 데이터프레임 생성
users_df = spark.createDataFrame(user_lst, schema)

In [33]:
users_df.show()

+------+
|userID|
+------+
|     1|
+------+



In [34]:
# [+] recommendForUserSubset(): 특정 유저 그룹에 대한 아이템 추천
user_recs = model.recommendForUserSubset(users_df,5)

In [35]:
user_recs.show()

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{69699, 5.407512...|
+------+--------------------+



In [36]:
# 추천 결과를 Python 객체로 받아오기
movies_lst = user_recs.collect()[0].recommendations

In [37]:
movies_lst

[Row(movieId=69699, rating=5.407512187957764),
 Row(movieId=26758, rating=5.407512187957764),
 Row(movieId=8235, rating=5.206705570220947),
 Row(movieId=7371, rating=4.9619669914245605),
 Row(movieId=32591, rating=4.914216041564941)]

In [38]:
# movies_lst에 대한 데이터프레임 생성
recs_df = spark.createDataFrame(movies_lst)
recs_df.show()

+-------+------------------+
|movieId|            rating|
+-------+------------------+
|  69699| 5.407512187957764|
|  26758| 5.407512187957764|
|   8235| 5.206705570220947|
|   7371|4.9619669914245605|
|  32591| 4.914216041564941|
+-------+------------------+



In [42]:
# [+] 영화 데이터에 대한 데이터프레임 생성

file = 'movies.csv'

movies_df = spark.read.csv(path+file, inferSchema = True, header=True)
movies_df.show()

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
|      6|         Heat (1995)|Action|Crime|Thri...|
|      7|      Sabrina (1995)|      Comedy|Romance|
|      8| Tom and Huck (1995)|  Adventure|Children|
|      9| Sudden Death (1995)|              Action|
|     10|    GoldenEye (1995)|Action|Adventure|...|
|     11|American Presiden...|Comedy|Drama|Romance|
|     12|Dracula: Dead and...|       Comedy|Horror|
|     13|        Balto (1995)|Adventure|Animati...|
|     14|        Nixon (1995)|               Drama|
|     15|Cutthroat Island ...|Action|Adventure|...|
|     16|       Casino (1995)|         Crime|Drama|
|     17|Sen

In [45]:
# [+] recs_df, movies_df 에 대한 Temporary View 생성
recs_df.createOrReplaceTempView('recommendation')
movies_df.createOrReplaceTempView('movies')

In [ ]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
spark.sql("""
    SELECT *
    FROM movies JOIN recommendations
    ON movies.movieID = recommendations.movieID
    ORDER BY rating DESC
""").show()

### **4. 유저 별 영화 추천 서비스를 간단하게 구현하기**
1. SQL문 작성
2. 영화 추천 함수 작성
3. 영화 추천 테스트

In [ ]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
query = """
SELECT * 
FROM movies JOIN recommendations 
ON movies.movieID = recommendations.movieID
ORDER BY rating DESC
"""

In [ ]:
# 입력된 유저에 대한 영화 추천 함수
def get_recommendations(user_id, num_recs):
    users_df = spark.createDataFrame([user_id], IntegerType()).toDF('userID')
    users_recs_df = model.recommendForUserSubset(users_df, num_recs)
    recs_lst = users_recs_df.collect()[0].recommendations
    recs_df = spark.createDataFrame(recs_lst)
    recs_df.createOrReplaceTempView('recommendations')
    recommended_movies = spark.sql(query)
    
    return recommended_movies

In [ ]:
# 1번 유저에 대한 영화 5개 추천
recs = get_recommendations(1, 5)

In [ ]:
# 추천 결과 출력
recs.show()

In [ ]:
# toPandas(): Pandas 데이터프레임으로 출력
recs.toPandas()